# Kaggle 競賽初始化流程
1. 列出 playground 競賽（依 deadline 排序）
2. 選擇競賽 → 建立資料夾 → 下載 → 解壓 → 複製 template

In [1]:
import subprocess, os, shutil, sys, zipfile, json
from pathlib import Path

# ── 設定 ──
PROJECT_ROOT = Path.cwd().resolve()  # AgentML 根目錄
TEMPLATE_DIR = PROJECT_ROOT / '.agents' / 'skills' / 'agentml' / 'assets' / 'project-template'
LEGACY_TEMPLATE_DIR = PROJECT_ROOT / 'templates'  # fallback for older clones
if not TEMPLATE_DIR.exists():
    print(f'Warning: skill payload not found, falling back to {LEGACY_TEMPLATE_DIR}')
    TEMPLATE_DIR = LEGACY_TEMPLATE_DIR
PROJECTS_DIR = PROJECT_ROOT / 'projects'
PROJECTS_DIR.mkdir(exist_ok=True)

# ── Kaggle CLI ──
# 不可依賴 PATH 上的 kaggle：改用這個 kernel 自己的 python -m kaggle
# 舊寫法 subprocess.run(['kaggle', ...]) 在 PATH 沒有 env 的 Scripts 時
# 會拋 FileNotFoundError: [WinError 2] The system cannot find the file specified
KAGGLE_CMD = [sys.executable, '-m', 'kaggle']

def run_kaggle(*args, **kwargs):
    """執行 Kaggle CLI（等價於 `kaggle ...`），不需要 kaggle 在 PATH 上。"""
    return subprocess.run(KAGGLE_CMD + list(args), **kwargs)


## 1. 列出 Playground 競賽

In [2]:
# 列出 playground 競賽，依最新 deadline 排序
result = run_kaggle(
    'competitions', 'list', '--category', 'playground', '--sort-by', 'latestDeadline',
    capture_output=True, text=True,
)
print(result.stdout)

# 只顯示真正的錯誤（忽略「outdated kaggle version」升級提醒）
err_msg = '\n'.join(
    line for line in (result.stderr or '').splitlines() if 'outdated' not in line
)
if err_msg:
    print('[STDERR]', err_msg)


ref                                                                                          deadline             category       reward  teamCount  userHasEntered  
-------------------------------------------------------------------------------------------  -------------------  ----------  ---------  ---------  --------------  
https://www.kaggle.com/competitions/the-pokemon-company-ptcg-ai-battle-challenge-playground  2027-01-08 23:59:59  Playground       Swag         39           False  
https://www.kaggle.com/competitions/playground-series-s6e9                                   2026-09-30 23:59:00  Playground       Swag       3513           False  
https://www.kaggle.com/competitions/playground-series-s6e8                                   2026-08-31 23:59:00  Playground       Swag       3531           False  
https://www.kaggle.com/competitions/autonomous-agent-prediction-beta                         2026-08-06 23:59:00  Playground       Swag        570           False  
https://ww

## 2. 選擇競賽並下載

In [3]:
# ⬇️ 在這裡輸入競賽 ID（例如 playground-series-s6e5）
COMPETITION_ID = 'playground-series-s6e9'  # ← 手動填

if not COMPETITION_ID:
    raise ValueError('請在上一格填入 COMPETITION_ID')

In [4]:
COMP_DIR = PROJECTS_DIR / COMPETITION_ID
print(f'正在建立專案資料夾: {COMP_DIR}')

正在建立專案資料夾: C:\Users\Raylao\Desktop\python_project\github\AgentML\projects\playground-series-s6e9


In [6]:
# ── 建立目錄結構 ──
COMP_DIR = PROJECTS_DIR / COMPETITION_ID
RAW_DIR = COMP_DIR / 'data' / 'raw'
RAW_DIR.mkdir(parents=True, exist_ok=True)
print(f'✅ 建立: {RAW_DIR}')

# ── 下載 ZIP（直接在 raw 目錄） ──
os.chdir(str(RAW_DIR))
run_kaggle('competitions', 'download', '-c', COMPETITION_ID, check=True)
print('✅ 下載完成')

# ── 解壓 ──
zip_path = RAW_DIR / f'{COMPETITION_ID}.zip'
if zip_path.exists():
    with zipfile.ZipFile(str(zip_path), 'r') as zf:
        zf.extractall(str(RAW_DIR))
    zip_path.unlink()  # 刪除 zip
    print('✅ 解壓完成，已刪除 zip')
else:
    print('⚠️ 找不到 zip，可能已解壓過')


✅ 建立: C:\Users\Raylao\Desktop\python_project\github\AgentML\projects\playground-series-s6e9\data\raw
✅ 下載完成
✅ 解壓完成，已刪除 zip


## 3. 複製 Templates

In [7]:
print('✅ 競賽資料夾結構已建立完成！')
print(TEMPLATE_DIR)
print(COMP_DIR)

✅ 競賽資料夾結構已建立完成！
C:\Users\Raylao\Desktop\python_project\github\AgentML\templates
C:\Users\Raylao\Desktop\python_project\github\AgentML\projects\playground-series-s6e9


In [8]:
# ── 複製 skill template payload 到競賽目錄（跳過 data/） ──
def copy_template(src: Path, dst: Path):
    """複製 template 檔案/資料夾，跳過 data/ 避免蓋掉已下載的資料"""
    if src.name == 'data':
        return
    if src.is_file():
        shutil.copy2(str(src), str(dst))
    elif src.is_dir():
        dst.mkdir(parents=True, exist_ok=True)
        for item in src.iterdir():
            copy_template(item, dst / item.name)

copy_template(TEMPLATE_DIR, COMP_DIR)
print(f'✅ Templates 已複製到 {COMP_DIR}')

# ── 回到專案根目錄 ──
os.chdir(str(PROJECT_ROOT))
print(f'✅ 回到專案根目錄: {PROJECT_ROOT}')

# ── 最終確認 ──
print('\n📁 目錄結構:')
for p in sorted(COMP_DIR.rglob('*')):
    if p.is_file():
        print(f'  {p.relative_to(COMP_DIR)}')

✅ Templates 已複製到 C:\Users\Raylao\Desktop\python_project\github\AgentML\projects\playground-series-s6e9
✅ 回到專案根目錄: C:\Users\Raylao\Desktop\python_project\github\AgentML

📁 目錄結構:
  AGENT_RULES.md
  configs\baseline.yaml
  configs\README.md
  configs\search_space.yaml
  data\raw\sample_submission.csv
  data\raw\test.csv
  data\raw\train.csv
  doc\00_problem_statement.md
  doc\01_data_card.md
  doc\02_eda.md
  doc\03_features_engineering.md
  doc\04_cv_strategy.md
  doc\05_ensemble.md
  doc\06_experiment_log.md
  doc\07_modeling.md
  doc\08_deployment_or_submission.md
  memory\debugging.md
  memory\eda_report.md
  memory\MEMORY.md
  program.md
  README.md
  results.json
  src\data.py
  src\eda.py
  src\ensemble.py
  src\evaluate.py
  src\features.py
  src\infer.py
  src\infer_ensemble.py
  src\train.py
  src\train_multi_model.py
